In [ ]:
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
import warnings

df = pd.read_csv('insurance.csv')

## EDA
df.isnull().sum() - check null values in each column

df.shape                        --- tells dimension of dataset

df.info()                       --- tells data abt the columns

df.describe()                   --- tells stats about the cols

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()       #check for null values

In [ ]:
num_cols = ['age', 'bmi', 'children', 'charges']
i = 1
plt.figure(figsize=(20,20))
for col in num_cols:
    plt.subplot(4,2,i)
    sb.histplot(df[col], kde = True, bins=20)
    i += 1
    plt.subplot(4,2,i)
    sb.boxplot(x=df[col],hue=df['sex'])
    i += 1

In [ ]:
categ_cols = ['sex', 'smoker', 'region']
i = 1
plt.figure(figsize=(10,6))
for col in categ_cols:
    plt.subplot(2,2,i)
    sb.countplot(x=col, data=df, palette='rainbow')
    i += 1

In [ ]:
# HEATMAP FOR CORRELATION -- NUMERIC ONLY
sb.heatmap(df.corr(numeric_only=True), annot=True)

### DATA CLEANING AND PREPROCESSING
##### handle Null values, duplicates
- isnull()                                  --                    --- for checking Null values
- drop_duplicates(inplace=true)             --                    --- removing duplicated rows
- df['col'] = df['col'].map({dictionary_to_map_val})    --        --- to label encoding

- pd.get_dummies(df, columns = [c1, c2], dropfirst = true)        --- to one hot encode , dropfirst removes one type, making it the obvios choice. If not all other, then the exculed one is the type
- df.astype(int)                                                 --- code in from of 0,1

In [ ]:
# No null values in current dataset
# Duplicate removal

df.drop_duplicates(inplace=True)

# renaming
df.rename(columns={
    'sex' : 'isMale',
    'smoker' : 'isSmoker',
}, inplace=True)

# label encoding 
df['isMale']       = df['isMale'].map({'male': 1, 'female': 0})
df['isSmoker']    = df['isSmoker'].map({"yes": 1, "no": 0})

In [ ]:
# ONE HOT ENCODING
df = pd.get_dummies(df, columns=['region'], drop_first=True)
df = df.astype(int)
df.head()

# Fature Engineering & Extraction
usually done on input variables only
##### creating col for subdivisions of a col
- create bins using cut functions
- classify bins and add labels
## Feature Scaling
needed for models which get affected by weights, ex: linear reg, logistic reg, etc

- StandardScaler() --- -- -- Standardization pupose
- StandardScaler().fit_transform(data) ---- -- - transform to standardized data 
## Feature Extraction
Done to select some features 
- from scipy.stats import pearsonr -- to check correlation against target,( here charges)

In [15]:
# subdividing and adding cols acc to BMI
df['body_category'] = pd.cut(
    df['bmi'],
    bins = [0, 18.5, 24.9, 29.9, float('inf')], 
    labels= ['underweight', 'normal', 'overweight', 'obese']
)
df.head()


,age,isMale,bmi,children,isSmoker,charges,region_northwest,region_southeast,region_southwest,body_category_underweight,body_category_normal,body_category_overweight,body_category_obese,body_category
0,-1.440418,0,-0.517949,-0.909234,1,16884,0,0,1,0,0,1,0,NaN
1,-1.511647,1,0.462463,-0.079442,0,1725,0,1,0,0,0,0,1,underweight
2,-0.799350,1,0.462463,1.580143,0,4449,0,1,0,0,0,0,1,underweight
3,-0.443201,1,-1.334960,-0.909234,0,21984,1,0,0,0,1,0,0,NaN
4,-0.514431,1,-0.354547,-0.909234,0,3866,1,0,0,0,0,1,0,NaN


In [ ]:
# one hot encoding on body category
df = pd.get_dummies(df, columns=['body_category'], drop_first=False)
df = df.astype(int)
df.head()

In [ ]:
#Feature Scaling

from sklearn.preprocessing import StandardScaler
std_cols = ['age', 'bmi', 'children']
df[std_cols] = StandardScaler().fit_transform(df[std_cols])    #using STANDARDSCALAR CLASS & ITS METHOD
df 

In [17]:
# Feature Extraction

from scipy.stats import pearsonr     # to check correlation against target, here charges
selected_features = [
    'age', 'bmi', 'children', 'isMale', 'isSmoker',
    'region_northwest', 'region_southeast', 'region_southwest',
    'body_category_normal', 'body_category_overweight', 'body_category_obese'
]
# create a dictionary of correlations against charges
correlations = {
    feature: pearsonr(df[feature], df['charges'])[0]
    for feature in selected_features
}

#store the correlation dict in a dataframe
correlation_df = pd.DataFrame(list(correlations.items()), columns=['Feature', 'Pearson Correlation'])
correlation_df.sort_values(by='Pearson Correlation', ascending=False)

,Feature,Pearson Correlation
4,isSmoker,0.787234
0,age,0.298309
10,body_category_obese,0.200348
1,bmi,0.196236
6,region_southeast,0.073577
2,children,0.067390
3,isMale,0.058046
5,region_northwest,-0.038695
7,region_southwest,-0.043637
8,body_category_normal,-0.104042
